# Assignment 4
The goal of this assigment is to experiment with word vectors and understand pretrained word embeddings.  
The word embeddings used are part of the GloVe Wikipedia 2014 + Gigaword 5 dataset.

In [1]:
import numpy as np

path = "datasets/glove.6B.100d.txt"

data = {}

with open(path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype='float32')
        data[word] = vector

The first vector being tested is "woman - man + king".  
This could be taken to mean just "woman + king" and the expected result is "queen".

In [2]:
man = data['man']
woman = data['woman']
king = data['king']
vector_expression = woman - man + king

[-0.10231996 -0.81294     0.10211001  0.985924    0.34218282  1.09095
 -0.48913    -0.05616698 -0.21029997 -1.02996    -0.86851     0.36786997
  0.01960999  0.59259    -0.231901   -1.016919   -0.012184   -1.17194
 -0.52329     0.60645    -0.98537004 -1.001028    0.48913902  0.630072
  0.58224     0.15908998  0.43684998 -1.25351     0.97054005 -0.06552899
  0.733763    0.44219002  1.2091839   0.19698    -0.15948     0.34364
 -0.46222997  0.33772     0.14792703 -0.24959499 -0.77093005  0.522717
 -0.12830001 -0.91881    -0.01755    -0.44041002 -0.52656496  0.33734798
  0.60639    -0.45067    -0.04158002  0.08408298  1.31456     0.67737997
 -0.24316001 -2.071      -0.60648996  0.19710997  0.63567     0.07819999
  0.49161002  0.08172001  0.708557    0.201938    0.5155501  -0.23025298
 -0.40473     0.39212003 -0.5093     -0.139153    0.21609999 -0.628671
  0.08894001  0.49167    -0.06637001  0.76095    -0.19442001  0.41131
 -1.04476    -0.14801991 -0.098355   -0.25115     0.808957    0.36312

In [7]:
from sklearn.metrics.pairwise import cosine_similarity 

# Function to find similar words 
def find_similar_words_to_vector(target_vector, exclude={}, top_n=5): 
    similarities = []
    target_vec = target_vector.reshape(1, -1)

    for word, vec in data.items():
        if word in exclude:
            continue
            
        sim = cosine_similarity(target_vec, vec.reshape(1, -1))
        similarities.append((word, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_n]

# exclude the words that were used in the vector
to_exclude = {'man', 'woman', 'king'}
result = find_similar_words_to_vector(vector_expression, to_exclude)

In [8]:
for word, score in result:
    print(word, score)

queen [[0.78344136]]
monarch [[0.69338024]]
throne [[0.6833109]]
daughter [[0.6809083]]
prince [[0.6713142]]


As expected, "queen" was the most likely match according to cosine similarity.  
The other words clearly lean on either the royalty aspect (king) or femininity (woman).  

Next is a simple expression "slow" + "heavy".  
The expected results could be either objects / animals, or adjectives similar to them.

In [9]:
vector_expression = data['slow'] + data['heavy']
to_exclude = {'slow', 'heavy'}
result = find_similar_words_to_vector(vector_expression, to_exclude)

In [10]:
for word, score in result:
    print(word, score)

fast [[0.73089963]]
low [[0.7109994]]
hard [[0.6993772]]
pressure [[0.69515765]]
too [[0.6949615]]


Interestingly, no objects are among the results. The words are either adjectives or concepts.  
"fast" is most likely included on the top because of its similarity to "slow".  
"low" is probably counted as a synonym of "slow" while "hard" most likely is related to "heavy".  
To make it more interesting, "fast" will be removed from the next equation.

In [17]:
vector_expression = data['slow'] - data['fast'] + data['heavy']
to_exclude = {'slow', 'fast', 'heavy'}
result = find_similar_words_to_vector(vector_expression, to_exclude)

In [18]:
for word, score in result:
    print(word, score)

caused [[0.6343254]]
due [[0.6323474]]
rain [[0.62872994]]
massive [[0.60701054]]
fog [[0.6069195]]


Now the results are quite interesting, and somewhat incoherent.  
The word "massive" is closest to what the equation slow - fast + heavy might reasonably produce as a semantics experiment.  
'caused', 'due' and 'rain' seem very strange, but 'fog' does in actually somewhat fit in its own way. 

For the last experiment, the function will analyze the expression "animal" + "strong" - "start".  
This isn't expected to produce coherent results since the concepts are a bit abstract.

In [15]:
vector_expression = data['animal'] + data['strong'] - data['smart']
to_exclude = {'animal', 'strong', 'smart'}
result = find_similar_words_to_vector(vector_expression, to_exclude)

In [16]:
for word, score in result:
    print(word, score)

human [[0.65822446]]
animals [[0.6305089]]
concern [[0.59485316]]
cause [[0.58471906]]
nature [[0.5843125]]


In its own way, "human" as the top result does make sense.  
The word "cause" appeared again, which might be caused by the two adjectives in the expression.  
Overall, the expressions seem to find words that are grouped close to either of the words involved.